# Sleep-EDF: следующий pretrained-этап

Сначала A–D: `RUN_GROUP="cheap"`, `ALLOW_TRAINING=False`. Сначала Run All с `SMOKE_ONLY=True`, затем `False`.
В режиме cheap выполняются 020–043, кроме дорогих дополнительных HF8 ансамблей 034/036.
Их можно отдельно выбрать через `RUN_GROUP="hf8"`. Модели проходят val → test; после каждой готов submission.

Подключите исходный **full_psg_v1** и **прошлые результаты 019** как Input: распакованный artefacts или скачанные pretrained ZIP.
Табличные кэши не нужны. Internet On, GPU T4 ×2. Код должен быть опубликован по CODE_REF; приватный checkout использует существующий Secret.

019 val переиспользуется после проверки provenance и ID. Недостающий test вычисляется один раз и сохраняется с submission.
Каждый checkpoint, канал и split кэшируется отдельно; фиксированные ансамбли и повторный запуск используют сохранённые вероятности.
HF soft voting — softmax среднего logits, hard — доли голосов, без изменения sampling rate. Оба варианта получают один backbone forward.
Viterbi — дискриминативный переходный штраф, обученный только по train labels. Его submission использует MAP labels,
а probabilities — forward/backward marginals; эти два argmax могут различаться.

E/F запускаются отдельно: `RUN_GROUP="correction"` или `"finetune"` и явное `ALLOW_TRAINING=True`.
Корректор делает grouped CV и full-train refit, fine-tuning — subject holdout с early stopping.
Official val/test не используются для обучения. Каждая T4 — отдельная GPU 16 ГБ; один процесс использует DEVICE.

Smoke на одной записи включает полный рабочий блок AnySleep; это не оценка качества. Fine-tuning требует отдельного GPU smoke с проверкой peak memory.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPO_URL = "https://github.com/revlze/sleep-edf-classification.git"
CODE_REF = "feat/pretrained"  # После публикации лучше закрепить SHA коммита.
PROJECT_ROOT = Path("/tmp/sleep-edf-pretrained-stage2")
OUTPUT_ROOT = Path("/kaggle/working/artefacts")
CHECKPOINT_CACHE = Path("/kaggle/working/pretrained-checkpoints")
SIGNAL_CACHE = Path("/kaggle/working/neural-signal-cache")
TRAINED_ROOT = Path("/kaggle/working/trained-checkpoints")
CHECKPOINT_ROOT = None  # Необязательный Input с официальными весами/исходниками.
RESULT_INPUTS = []  # Автопоиск прежнего 019: ZIP или каталог artefacts в Input.
RESUME_ROOT = (
    None  # Прошлый распакованный OUTPUT_ROOT; не смешивать разные версии кода.
)
RESUME_TRAINED_ROOT = None  # Для E/F: отдельные best/last checkpoints прошлого запуска.

RUN_GROUP = "cheap"  # cheap / screening / fusion / hf4 / hf8 / postprocessing / tiny / correction / finetune / baseline / custom
CUSTOM_IDS = []
SMOKE_ONLY = True  # False: сначала smoke, затем полный val → test у успешных smoke.
SMOKE_EPOCHS = 399
COMPLETE_019_TEST = (
    True  # Исходный val 019 переиспользуется; недостающий test вычисляется один раз.
)
ALLOW_TRAINING = False  # Только явное True + correction/finetune разрешает E/F.
EXPORT_TRAINED_CHECKPOINTS = (
    True  # Отдельный ZIP весов после E/F; вне компактного ZIP результатов.
)
DEVICE = "cuda:0"  # Одна T4 на процесс; независимую группу можно запустить на cuda:1.
THREADS = max(1, (os.cpu_count() or 2) - 1)

GROUPS = {
    "cheap": [i for i in range(20, 44) if i not in (34, 36)],
    "screening": [20, 21, 22, 23, 29, 30, 31, 32],
    "fusion": [24, 25, 26, 27, 28],
    "hf4": [29, 31, 33, 35],
    "hf8": [30, 32, 34, 36],  # Дополнительные HF8 checkpoints: отдельный этап.
    "postprocessing": [37, 38, 39, 40],
    "tiny": [41, 42, 43],
    "correction": [44, 45],
    "finetune": [46, 47, 48],
    "baseline": [],
}
CONFIG_IDS = (
    CUSTOM_IDS
    if RUN_GROUP == "custom"
    else [f"{i:03d}_pretrained" for i in GROUPS[RUN_GROUP]]
)
if any(int(name[:3]) >= 44 for name in CONFIG_IDS) and not ALLOW_TRAINING:
    raise RuntimeError(
        "E/F обучаются только после явного ALLOW_TRAINING=True. Сначала выполните A–D."
    )


def checkout_code():
    if PROJECT_ROOT.exists():
        current_ref = subprocess.check_output(
            ["git", "config", "sleepedf.bootstrapRef"], cwd=PROJECT_ROOT, text=True
        ).strip()
        if current_ref != CODE_REF:
            raise RuntimeError(
                "Для другого CODE_REF задайте новый PROJECT_ROOT в /tmp."
            )
        print(
            "Используется существующий checkout; для обновления ветки задайте новый PROJECT_ROOT."
        )
        return
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    with tempfile.TemporaryDirectory(prefix="sleep-git-") as temporary:
        askpass = Path(temporary) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\nimport os, sys\n"
            'print("x-access-token" if "Username" in sys.argv[1] else os.environ["SLEEP_GITHUB_TOKEN"])\n'
        )
        askpass.chmod(0o700)
        env = {
            **os.environ,
            "GIT_ASKPASS": str(askpass),
            "GIT_TERMINAL_PROMPT": "0",
            "SLEEP_GITHUB_TOKEN": token,
        }
        subprocess.run(
            ["git", "init", "--quiet", "--initial-branch=bootstrap", str(PROJECT_ROOT)],
            check=True,
        )
        subprocess.run(
            ["git", "remote", "add", "origin", REPO_URL], cwd=PROJECT_ROOT, check=True
        )
        subprocess.run(
            [
                "git",
                "-c",
                "credential.helper=",
                "fetch",
                "--quiet",
                "--depth",
                "1",
                "origin",
                CODE_REF,
            ],
            cwd=PROJECT_ROOT,
            env=env,
            check=True,
        )
        subprocess.run(
            ["git", "checkout", "--quiet", "--detach", "FETCH_HEAD"],
            cwd=PROJECT_ROOT,
            check=True,
        )
        subprocess.run(
            ["git", "config", "sleepedf.bootstrapRef", CODE_REF],
            cwd=PROJECT_ROOT,
            check=True,
        )
    del token


checkout_code()
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
print("Код:", commit)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.notebook import run_command  # noqa: E402

# Устанавливаем только инструмент uv; зависимости проекта устанавливает uv.lock.

UV_VERSION = "0.12.15"
if shutil.which("uv") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", f"uv=={UV_VERSION}"], check=True
    )
UV = shutil.which("uv")
ENV = {
    "UV_PROJECT_ENVIRONMENT": "/tmp/sleep-edf-pretrained-venv",
    "UV_CACHE_DIR": "/tmp/sleep-pretrained-uv-cache",
    "UV_PYTHON_INSTALL_DIR": "/tmp/sleep-pretrained-python",
    "PYTHONPATH": str(PROJECT_ROOT),
}
UV_PROJECT = PROJECT_ROOT / "environments/pretrained"
print("Ядро notebook:", sys.version)
subprocess.run(["nvidia-smi"], check=True)
run_command(
    [
        UV,
        "sync",
        "--project",
        str(UV_PROJECT),
        "--locked",
        "--no-dev",
        "--python",
        "3.12.12",
    ],
    cwd=PROJECT_ROOT,
    env=ENV,
)
BASE = [
    UV,
    "run",
    "--project",
    str(UV_PROJECT),
    "--locked",
    "--no-dev",
    "python",
    "-m",
    "src.pretrained_series",
]
run_command(
    [
        UV,
        "run",
        "--project",
        str(UV_PROJECT),
        "--locked",
        "--no-dev",
        "python",
        "-c",
        "from src.pretrained import hardware; import json; print(json.dumps(hardware("
        + repr(DEVICE)
        + "),indent=2))",
    ],
    cwd=PROJECT_ROOT,
    env=ENV,
)

In [ ]:
roots = [
    p.parent
    for p in Path("/kaggle/input").rglob("labels.csv")
    if (p.parent / "epochs/val").is_dir()
    and (p.parent / "epochs/test").is_dir()
    and (p.parent / "MANIFEST.json").is_file()
]
if len(roots) != 1:
    raise RuntimeError(f"Нужен один исходный full_psg_v1; найдено: {roots}")
DATA_ROOT = roots[0]
if not RESULT_INPUTS:
    candidates = [
        p.parent.parent
        for p in Path("/kaggle/input").rglob("val_predictions.npz")
        if p.parent.name == "019_pretrained" and "smoke" not in p.parts
    ]
    candidates += list(Path("/kaggle/input").rglob("sleep-edf-pretrained-*.zip"))
    candidates += list(Path("/kaggle/input").rglob("sleep-edf-series-*.zip"))
    RESULT_INPUTS = sorted(set(candidates))
if (
    not RESULT_INPUTS
    and RESUME_ROOT is None
    and not (OUTPUT_ROOT / "predictions").exists()
):
    raise RuntimeError(
        "Подключите прошлые результаты 019 как Input (каталог artefacts или скачанный ZIP), чтобы не повторять исходный val inference."
    )
for previous, target in [
    (RESUME_ROOT, OUTPUT_ROOT),
    (RESUME_TRAINED_ROOT, TRAINED_ROOT),
]:
    if previous is not None:
        if not target.exists():
            shutil.copytree(Path(previous), target)
        else:
            print(
                f"Продолжение из существующего {target}; {previous} повторно не копируется."
            )
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Данные:", DATA_ROOT)
print("Ранее сохранённый 019:", [str(p) for p in RESULT_INPUTS])
print(
    "Конфигурации:",
    CONFIG_IDS,
    "SMOKE_ONLY:",
    SMOKE_ONLY,
    "ALLOW_TRAINING:",
    ALLOW_TRAINING,
)

COMMON = [
    "--data-root",
    str(DATA_ROOT),
    "--output-root",
    str(OUTPUT_ROOT),
    "--checkpoint-cache",
    str(CHECKPOINT_CACHE),
    "--signal-cache",
    str(SIGNAL_CACHE),
    "--trained-root",
    str(TRAINED_ROOT),
    "--device",
    DEVICE,
    "--threads",
    str(THREADS),
]
for previous in RESULT_INPUTS:
    COMMON += ["--result-input", str(previous)]
if CHECKPOINT_ROOT is not None:
    COMMON += ["--checkpoint-root", str(CHECKPOINT_ROOT)]
if ALLOW_TRAINING:
    COMMON += ["--allow-training"]
if COMPLETE_019_TEST or RUN_GROUP == "baseline":
    COMMON += ["--complete-baseline"]

In [ ]:
import pandas as pd


def run_batch(names, smoke):
    command = [
        *BASE,
        *[str(PROJECT_ROOT / "configs" / f"{name}.yaml") for name in names],
        *COMMON,
    ]
    if smoke:
        command += ["--smoke", "--smoke-epochs", str(SMOKE_EPOCHS)]
    import time

    started_ns = time.time_ns()
    process_failed = False
    try:
        run_command(command, cwd=PROJECT_ROOT, env=ENV)
    except subprocess.CalledProcessError:
        process_failed = True
        print(
            "Некоторые эксперименты завершились ошибкой; готовые результаты сохранены."
        )
    namespace = OUTPUT_ROOT / "smoke" if smoke else OUTPUT_ROOT
    failures_path = namespace / "series_failures.json"
    fresh_failures = (
        failures_path.exists() and failures_path.stat().st_mtime_ns >= started_ns
    )
    if process_failed and not fresh_failures:
        print(
            "Процесс прерван до итогового отчёта. Полный запуск этой группы пропущен; результаты доступны для экспорта."
        )
        return []
    failures = json.loads(failures_path.read_text()) if fresh_failures else []
    for item in failures:
        print(item["experiment_id"], item["error"])
    failed_ids = {item["experiment_id"] for item in failures}
    return [
        name
        for name in names
        if name not in failed_ids and (namespace / name / "complete.json").exists()
    ]


passed = run_batch(CONFIG_IDS, smoke=True)
if not SMOKE_ONLY:
    run_batch(passed, smoke=False)

namespace = OUTPUT_ROOT / "smoke" if SMOKE_ONLY else OUTPUT_ROOT
rows = []
for name in CONFIG_IDS:
    path = namespace / name / "metrics.json"
    if (namespace / name / "complete.json").exists() and path.exists():
        m = json.loads(path.read_text())
        rows.append(
            {
                "experiment": name,
                "mode": "SMOKE" if m["smoke"] else "VAL+TEST",
                "val_macro_f1": m["validation"]["macro_f1"],
                "test_epochs": m["test_epochs"],
                "request_seconds": round(m["inference_seconds"], 1),
                "peak_GPU_GiB": round(m["peak_gpu_bytes"] / 2**30, 2)
                if m["peak_gpu_bytes"] is not None
                else None,
                "submission": "submission_smoke.csv"
                if m["smoke"]
                else "submission.csv",
            }
        )
    else:
        rows.append({"experiment": name, "mode": "не завершён"})

display(pd.DataFrame(rows))

Готовые эксперименты и отдельные записи восстанавливаются автоматически. Ошибка одной конфигурации не удаляет остальные.
Smoke не пишет experiments.csv и создаёт только **submission_smoke.csv**, который нельзя отправлять в Kaggle.
Обычный `submission.csv` создаётся после проверки полного test и точного порядка sample_submission.
Результаты выгружаются компактным ZIP; сырые сигналы, signal cache и полные trained checkpoints в него не входят.
Для E/F веса best/last выгружаются второй отдельной ссылкой. Их нужно сохранить для возобновления обучения в новой сессии.


In [ ]:
from datetime import UTC, datetime
from zipfile import ZIP_DEFLATED, ZipFile

from IPython.display import FileLink, display

stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
archive = Path("/kaggle/working") / f"sleep-edf-series-{stamp}.zip"
run_command(
    [*BASE, "--output-root", str(OUTPUT_ROOT), "--export", str(archive)],
    cwd=PROJECT_ROOT,
    env=ENV,
)
display(
    FileLink(
        str(archive.relative_to(Path.cwd())),
        result_html_prefix="Результаты, вероятности и submissions: ",
    )
)
if ALLOW_TRAINING and EXPORT_TRAINED_CHECKPOINTS and TRAINED_ROOT.exists():
    weights_zip = Path("/kaggle/working") / f"sleep-edf-trained-{stamp}.zip"
    with ZipFile(weights_zip, "x", compression=ZIP_DEFLATED) as output:
        for path in sorted(TRAINED_ROOT.rglob("*")):
            if (
                path.is_file()
                and path.suffix in (".pt", ".json")
                and ".tmp." not in path.name
            ):
                output.write(path, path.relative_to(TRAINED_ROOT).as_posix())
    display(
        FileLink(
            str(weights_zip.relative_to(Path.cwd())),
            result_html_prefix="Отдельно best/last checkpoints для продолжения обучения: ",
        )
    )